In [0]:
%sql
CREATE CATALOG IF NOT EXISTS fmcg;

In [0]:
%sql
USE CATALOG fmcg;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS fmcg.gold;
CREATE SCHEMA IF NOT EXISTS fmcg.silver;
CREATE SCHEMA IF NOT EXISTS fmcg.bronze;

In [0]:
%sql
SELECT COUNT(*) FROM fmcg.gold.fact_orders;

Import Required Libraries

In [0]:
from pyspark.sql import functions as F

Define star and End Dates

In [0]:
# start date and end date
start_date = "2024-01-01"
end_date   = "2025-12-01"

In [0]:
# 1️⃣ Generate one row per month start between start_date and end_date
df = (
    spark.sql(f"""
        SELECT explode(
            sequence(
                to_date('{start_date}'),
                to_date('{end_date}'),
                interval 1 month
            )
        ) AS month_start_date
    """)
)


# 2️⃣ Add useful analytics columns
df = (
    df
    # Surrogate key at month grain
    .withColumn("date_key", F.date_format("month_start_date", "yyyyMM").cast("int"))
    .withColumn("year", F.year("month_start_date"))
    .withColumn("month_name", F.date_format("month_start_date", "MMMM"))
    .withColumn("month_short_name", F.date_format("month_start_date", "MMM"))
    .withColumn("quarter", F.concat(F.lit("Q"), F.quarter("month_start_date")))
    .withColumn("year_quarter", F.concat(F.col("year"), F.lit("-Q"), F.quarter("month_start_date")))
)

In [0]:
df.display()

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("fmcg.gold.dim_date")